In [1]:
import pandas as pd
import re

roll_no = input("Enter your college roll number: ").strip()

last_two = roll_no[-2:]
if len(last_two) != 2 or not last_two.isdigit():
    raise ValueError("Please enter a valid roll number ending with two digits.")

digits = [int(x) for x in last_two]
categories = ["billing", "account", "general"]

print("Last two digits:", digits)


Last two digits: [5, 0]


### Q1. Build the knowledge base



In [2]:
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

personalized = []

faq_by_category = {
    "billing": {
        "question": "how can i check my latest bill",
        "answer": "You can check your latest bill from the Billing section.",
        "keywords": "bill invoice amount"
    },
    "account": {
        "question": "how do i update my registered mobile number",
        "answer": "Go to Account Settings and update your registered mobile number.",
        "keywords": "mobile number update profile"
    },
    "general": {
        "question": "where can i find the help centre",
        "answer": "The Help Centre is available from the main menu.",
        "keywords": "help support centre"
    }
}

for digit in digits:
    category = categories[digit % 3]
    entry = faq_by_category[category].copy()
    entry["category"] = category
    personalized.append(entry)

entries = fixed_entries + personalized
df = pd.DataFrame(entries, columns=["question", "answer", "keywords", "category"])

print("Final 6-row DataFrame:")
display(df)


Final 6-row DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,where can i find the help centre,The Help Centre is available from the main menu.,help support centre,general
5,how can i check my latest bill,You can check your latest bill from the Billin...,bill invoice amount,billing


### Q2. Generate and score a hypothesis

In [3]:
def words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def score_faq(query, data):
    query_words = words(query)
    results = []

    for index, row in data.iterrows():
        faq_words = words(row["question"] + " " + row["keywords"])
        score = len(query_words & faq_words)
        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    results.sort(key=lambda x: x["score"], reverse=True)
    return pd.DataFrame(results)

query = input("\nEnter a query: ")
matches = score_faq(query, df)

if matches.empty:
    print("No matching FAQ found.")
else:
    display(matches)


No matching FAQ found.


### Q3. Find FAQs from the same category

In [4]:
def same_category(category_name, data):
    return data[data["category"].str.lower() == category_name.lower()]

chosen_category = personalized[0]["category"]
print("Category selected:", chosen_category)
display(same_category(chosen_category, df))


Category selected: general


,question,answer,keywords,category
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
4,where can i find the help centre,The Help Centre is available from the main menu.,help support centre,general


### Q4. Add a new keyword and save the CSV

In [5]:
entry_no = 0
print("Selected FAQ:")
print(df.loc[entry_no, "question"])

new_keyword = input("Enter a new keyword to add: ").strip().lower()

if new_keyword:
    current = df.loc[entry_no, "keywords"].split()
    if new_keyword not in current:
        current.append(new_keyword)
    df.loc[entry_no, "keywords"] = " ".join(current)

file_name = f"{roll_no}_faq_data.csv"
df.to_csv(file_name, index=False)

print("Updated entry:")
display(df.loc[[entry_no]])
print("Saved as:", file_name)


Selected FAQ:
what is the annual fee
Updated entry:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing


Saved as: 1024170250_faq_data.csv


### Q5. Number of FAQs in each category

In [6]:
category_counts = df.groupby("category").size()
print(category_counts)


category
account    1
billing    3
general    2
dtype: int64


### Q6. Handle ties for the highest score



In [7]:
def score_faq_with_ties(query, data):
    result = score_faq(query, data)

    if result.empty:
        print("No matching FAQ found.")
        return result

    highest = result["score"].max()
    best = result[result["score"] == highest]

    if len(best) > 1:
        print(f"Tie: {len(best)} entries have the highest score of {highest}.")
        display(best)
    else:
        print("Best match:")
        display(best)

    return result

print("Tie example")
tie_result = score_faq_with_ties("fee", df)

print("\nNon-tie example")
non_tie_result = score_faq_with_ties("password", df)


Tie example
Tie: 2 entries have the highest score of 1.


,index,question,answer,category,score
0,0,what is the annual fee,The annual fee is Rs 500.,billing,1
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1



Non-tie example
Best match:


,index,question,answer,category,score
0,1,how to reset password,Go to Settings > Reset Password.,account,1
